In [5]:
import torch
import gc

# Delete the model and processor variables if they exist
try:
    del model
    del processor
except NameError:
    pass

# Force Python's garbage collector to run
gc.collect()

# Empty the PyTorch CUDA cache
torch.cuda.empty_cache()

In [1]:
import os

# 1. Route all Hugging Face downloads to your spacious persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# 2. Keep the fix for the Xet network timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Hugging Face cache set to: {os.environ['HF_HOME']}")

Hugging Face cache set to: /workspace/huggingface_cache


In [4]:
!pip install -U torch torchvision torchaudio
!pip install -U transformers accelerate peft bitsandbytes trl datasets qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 29.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 31.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 42.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 31.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 35.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 34.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 29.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 36.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 34.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import json
import os
from datasets import Dataset

# Define RunPod workspace paths
WORKSPACE_DIR = "/workspace/"
MICRO_BATCH_PATH = os.path.join(WORKSPACE_DIR, "qwen25_vl_micro_batch_sft_optimized.json") # Update with your exact filename
LOCAL_IMAGE_DIR = os.path.join(WORKSPACE_DIR, "balanced_images_512")

with open(MICRO_BATCH_PATH, "r", encoding="utf-8") as f:
    micro_batch = json.load(f)

processed_data = []
for item in micro_batch:
    # Fix the Windows backslash to Linux forward slash
    raw_img_path = item["images"][0].replace("\\", "/")
    img_filename = os.path.basename(raw_img_path)
    
    # Create the absolute Linux path
    local_img_path = os.path.join(LOCAL_IMAGE_DIR, img_filename)
    
    processed_data.append({
        "id": item["id"],
        "messages": item["messages"],
        "images": [local_img_path]
    })

hf_dataset = Dataset.from_list(processed_data)
print(f"Loaded {len(hf_dataset)} samples successfully. First image path: {hf_dataset[0]['images'][0]}")

Loaded 300 samples successfully. First image path: /workspace/balanced_images_512/153341.png


In [3]:
import torch
from qwen_vl_utils import process_vision_info

class QwenDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        batch_messages = []
        
        for example in examples:
            raw_messages = example["messages"]
            img_path = example["images"][0] 
            
            formatted_messages = []
            image_added = False # Track if we've already added the image
            
            for msg in raw_messages:
                if msg["role"] == "user":
                    clean_text = msg["content"].replace("<|image_pad|>", "").strip()
                    
                    # Only append the image to the very first user prompt
                    if not image_added:
                        formatted_messages.append({
                            "role": "user",
                            "content": [
                                {"type": "image", "image": img_path},
                                {"type": "text", "text": clean_text}
                            ]
                        })
                        image_added = True
                    else:
                        formatted_messages.append({
                            "role": "user",
                            "content": clean_text
                        })
                else:
                    formatted_messages.append(msg)
            
            batch_messages.append(formatted_messages)

        # Apply chat template
        texts = [
            self.processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) 
            for msg in batch_messages
        ]
        
        image_inputs, video_inputs = process_vision_info(batch_messages)

        batch = self.processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )

        # ---> THE SILVER BULLET FIX <---
        # Force PyTorch autograd to track the vision tower graph during checkpointing
        if "pixel_values" in batch and batch["pixel_values"] is not None:
            batch["pixel_values"] = batch["pixel_values"].requires_grad_(True)
        if "pixel_values_videos" in batch and batch["pixel_values_videos"] is not None:
            batch["pixel_values_videos"] = batch["pixel_values_videos"].requires_grad_(True)

        # ---> SFT LABEL MASKING FIX <---
        labels = batch["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        # Mask everything except the assistant's responses
        # Qwen 2.5 uses <|im_start|>assistant\n to begin the response
        assistant_token_id = self.processor.tokenizer.convert_tokens_to_ids("<|im_start|>")
        
        for i in range(len(labels)):
            in_assistant_turn = False
            for j in range(len(labels[i])):
                token = labels[i][j]
                
                # If we hit <|im_start|>, check if the next tokens are "assistant"
                if token == assistant_token_id:
                    # Look ahead slightly to confirm it's the assistant turn
                    # (Skipping exact string matching for speed, relying on structural pattern)
                    in_assistant_turn = True
                    labels[i][j] = -100 # Mask the <|im_start|> token itself
                    continue
                
                # If we hit an <|im_end|> token, the assistant's turn is over
                if token == self.processor.tokenizer.convert_tokens_to_ids("<|im_end|>"):
                    in_assistant_turn = False
                    
                if not in_assistant_turn:
                    labels[i][j] = -100
                    
        batch["labels"] = labels
        
        return batch

In [4]:
import torch
# FIX 2: Import AutoModelForVision2Seq
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint
import os

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
processor = AutoProcessor.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    # FIX 1: Safely keep the vision tower in bf16 by skipping quantization
    llm_int8_skip_modules=["visual"] 
)

# FIX 2: Let AutoModel handle the Qwen2.5 architecture mapping
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Safely freeze the vision tower without altering dtypes
for name, param in model.named_parameters():
    if "visual" in name:
        param.requires_grad = False

model = prepare_model_for_kbit_training(model)

model.gradient_checkpointing_enable()
model.config.use_cache = False

lora_config = LoraConfig(
    r=64,                # <--- Increased from 32
    lora_alpha=128,      # <--- Scaled up with r
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Initialize your custom collator (assuming it's defined in the previous cell)
data_collator = QwenDataCollator(processor)

training_args = SFTConfig(
    output_dir="/workspace/qwen_lora_micro_4096_2",
    
    # --- EXTREME MEMORY SAVERS FOR 24GB VRAM ---
    per_device_train_batch_size=1,           # MUST be 1
    gradient_accumulation_steps=8,           # Simulates a batch size of 8
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",                # Offloads optimizer states
    # -------------------------------------------
    
    max_length=4096,                         # Safe limit for the 300 micro-batch
    learning_rate=2e-4,
    bf16=True,                               # Native bfloat16 for RTX 3090/4090/A5000
    fp16=False,
    max_steps=100,                           # Just enough to verify the loss drops
    logging_steps=10,
    save_steps=20,                           # Frequent saves in case of Spot interruption
    save_total_limit=3,
    remove_unused_columns=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    args=training_args,
    data_collator=data_collator,
    processing_class=processor,
)

last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    last_checkpoint = get_last_checkpoint(training_args.output_dir)

if last_checkpoint is not None:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
else:
    print("No checkpoint found. Starting training from scratch...")

trainer.train(resume_from_checkpoint=last_checkpoint)
trainer.model.save_pretrained("/workspace/qwen_lora_final")

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


No checkpoint found. Starting training from scratch...


Step,Training Loss
10,5.218274
20,2.967617
30,2.868421
40,3.138596
50,2.892177
60,2.827338
70,2.835548
80,2.862800
90,2.783097
100,2.975358
